In [32]:
from pathlib import Path
import pandas as pd
fold = Path(r"D:\BaiduNetdiskDownload\正则化基金数据\正则化基金数据")

nav = pd.read_feather(fold / "基金数据\交易日偏股型基金.feather")

SH = pd.read_feather(fold / "基金数据\CHINAMUTUALFUNDSTOCKPORTFOLIO.feather")

fm = pd.read_csv(fold / "宽基指数日行情\宽基指数收益率.csv")

fi = pd.read_feather(fold / "申万一级行业\申万一级行业_with_dailyreturn.feather")

fs = pd.read_feather(fold / "Barra_CNE5\Barra风格因子收益率.feather")

In [33]:
# # 数据质量检查
# import numpy as np
# from IPython.display import display

# def _date_quality(df, date_col):
#     s = df[date_col]
#     if np.issubdtype(s.dtype, np.number):
#         dt = pd.to_datetime(s.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
#     else:
#         dt = pd.to_datetime(s, errors='coerce')

#     d = dt.sort_values().dropna()
#     gaps = d.diff().dt.days.dropna()
#     return {
#         'date_col': date_col,
#         'dtype': str(s.dtype),
#         'missing_raw': int(s.isna().sum()),
#         'unparseable': int(dt.isna().sum() - s.isna().sum()),
#         'min_date': dt.min(),
#         'max_date': dt.max(),
#         'duplicated_dates': int(dt.duplicated().sum()),
#         'is_monotonic_increasing': bool(dt.is_monotonic_increasing),
#         'top_gap_days': gaps.value_counts().head(8).to_dict(),
#         'large_gap_count_gt_10d': int((gaps > 10).sum()),
#     }

# def _numeric_quality(df, cols, name):
#     rows = []
#     for col in cols:
#         x = pd.to_numeric(df[col], errors='coerce')
#         finite = np.isfinite(x)
#         rows.append({
#             'dataset': name,
#             'column': col,
#             'dtype': str(df[col].dtype),
#             'missing': int(df[col].isna().sum()),
#             'non_numeric_after_coerce': int(x.isna().sum() - df[col].isna().sum()),
#             'inf': int(np.isinf(x).sum()),
#             'min': x[finite].min() if finite.any() else np.nan,
#             'p01': x[finite].quantile(0.01) if finite.any() else np.nan,
#             'median': x[finite].median() if finite.any() else np.nan,
#             'p99': x[finite].quantile(0.99) if finite.any() else np.nan,
#             'max': x[finite].max() if finite.any() else np.nan,
#             'abs_gt_30pct': int((x.abs() > 0.30).sum()),
#             'abs_gt_100pct': int((x.abs() > 1.00).sum()),
#         })
#     return pd.DataFrame(rows)

# def _find_date_col(df):
#     candidates = ['time', 'date', '日期', 'ANN_DATE']
#     for c in candidates:
#         if c in df.columns:
#             return c
#     for c in df.columns:
#         if 'date' in str(c).lower():
#             return c
#     return None

# def _basic_quality(name, df):
#     date_col = _find_date_col(df)
#     print(f'\n===== {name} =====')
#     print('shape:', df.shape)
#     print('columns:', list(df.columns))
#     print('duplicate rows:', int(df.duplicated().sum()))
#     if date_col:
#         display(pd.DataFrame([_date_quality(df, date_col)]))
#     else:
#         print('没有识别到日期列')
#     return date_col

In [34]:
# # 数据质量检查：检查 nav / fm / fi / fs；按你的要求暂时不检查 SH。

# def _missing_summary(df, name):
#     miss = df.isna().sum()
#     out = pd.DataFrame({
#         'dataset': name,
#         'column': miss.index,
#         'missing': miss.values,
#         'missing_rate': miss.values / len(df),
#     })
#     return out[out['missing'] > 0].sort_values('missing_rate', ascending=False)

# def _check_nav(nav):
#     date_col = _basic_quality('nav', nav)
#     cols = list(nav.columns)
#     lower = {c: str(c).lower() for c in cols}

#     code_cols = [c for c in cols if lower[c] in ['code', 'fund_code', 'fundcode', '基金代码'] or 'fund' in lower[c] or '基金代码' in str(c)]
#     return_cols = [c for c in cols if lower[c] in ['return', 'ret', 'daily_return'] or 'return' in lower[c] or '收益' in str(c)]
#     wide_fund_cols = [c for c in cols if isinstance(c, str) and pd.Series([c]).str.match(r'^\\d{6}\\.OF$').iloc[0]]

#     print('\nnav 重要字段识别')
#     print('日期列:', date_col)
#     print('基金代码列:', code_cols if code_cols else '未发现长表基金代码列')
#     print('return列:', return_cols if return_cols else '未发现显式 return 列')
#     print('宽表基金代码列数量:', len(wide_fund_cols))

#     if code_cols:
#         for c in code_cols:
#             bad = nav.loc[~nav[c].astype(str).str.match(r'^\\d{6}\\.OF$', na=False), c].drop_duplicates().head(20)
#             dup_keys = int(nav.duplicated(subset=[c, date_col]).sum()) if date_col else np.nan
#             display(pd.DataFrame([{
#                 'code_col': c,
#                 'missing_code': int(nav[c].isna().sum()),
#                 'unique_code': int(nav[c].nunique(dropna=True)),
#                 'duplicated_code_date': dup_keys,
#                 'bad_code_sample': bad.tolist(),
#             }]))
#     elif wide_fund_cols:
#         bad_cols = [c for c in cols if c != date_col and c not in wide_fund_cols]
#         display(pd.DataFrame([{
#             'wide_fund_columns': len(wide_fund_cols),
#             'bad_non_fund_columns_except_date': bad_cols[:20],
#         }]))

#     if return_cols:
#         display(_numeric_quality(nav, return_cols, 'nav_return'))
#         if date_col and code_cols:
#             key_cols = code_cols + [date_col]
#             display(nav.loc[nav.duplicated(subset=key_cols, keep=False), key_cols + return_cols].head(20))
#     else:
#         print('提示：nav 没有识别到 return 列。如果这个文件还是净值列，需要先计算 return 后再检查收益率质量。')

# def _check_factor_or_market(name, df):
#     date_col = _basic_quality(name, df)
#     miss = _missing_summary(df, name)
#     print('\n缺失值列数:', len(miss), '总缺失:', int(df.isna().sum().sum()))
#     if len(miss):
#         display(miss.head(30))

#     data_cols = [c for c in df.columns if c != date_col]
#     numeric_cols = [c for c in data_cols if pd.api.types.is_numeric_dtype(df[c])]
#     non_numeric_cols = [c for c in data_cols if c not in numeric_cols]
#     print('数值列数量:', len(numeric_cols))
#     print('非数值列:', non_numeric_cols)

#     if numeric_cols:
#         q = _numeric_quality(df, numeric_cols, name)
#         display(q.sort_values(['abs_gt_100pct', 'abs_gt_30pct', 'missing'], ascending=False).head(40))

# for _name, _df in [('nav', nav), ('fm', fm), ('fi', fi), ('fs', fs)]:
#     if _name == 'nav':
#         _check_nav(_df)
#     else:
#         _check_factor_or_market(_name, _df)

In [35]:
# print(nav.columns)
# print(fm.columns)
# print(fi.columns)
# print(fs.columns)

In [36]:
# 设置回测时间区间
start_date = pd.to_datetime('20200630')
end_date = pd.to_datetime('20210629')
# 统一日期格式
nav['PRICE_DATE'] = pd.to_datetime(nav['PRICE_DATE'].astype(str))
fm['日期'] = pd.to_datetime(fm['日期'].astype(str))
fi['TRADE_DT'] = pd.to_datetime(fi['TRADE_DT'].astype(str))
fs['time'] = pd.to_datetime(fs['time'].astype(str))
# 统一日期区间
nav = nav[
    (nav['PRICE_DATE']>=start_date) & (nav['PRICE_DATE']<=end_date)
]
fm = fm[
    (fm['日期']>=start_date) & (fm['日期']<=end_date)
]
fi = fi[
    (fi['TRADE_DT']>=start_date) & (fi['TRADE_DT']<=end_date)
]
fs = fs[
    (fs['time']>=start_date) & (fs['time']<=end_date)
]

In [ ]:
# 筛选合格基金
codes = (
    nav.groupby('F_INFO_WINDCODE')
       .filter(lambda x: (
           x['PRICE_DATE'].nunique() > 120
       ) and (
           x['1M'].any() # 改为了any（）
       ))['F_INFO_WINDCODE']
       .unique()
)
len(codes)

3056

In [38]:
# 以宽基指数交易日为基准，补齐基金中间缺失日期，并合并基金收益率与因子收益率
trading_days = pd.Index(
    fm['日期'].dropna().drop_duplicates().sort_values(),
    name='PRICE_DATE'
)

# 只保留前面筛选出的基金；同一基金同一日如有重复，保留排序后的最后一条
nav_for_merge = (
    nav.loc[nav['F_INFO_WINDCODE'].isin(codes)]
       .sort_values(['F_INFO_WINDCODE', 'PRICE_DATE'])
       .drop_duplicates(['F_INFO_WINDCODE', 'PRICE_DATE'], keep='last')
       .copy()
)
original_fund_dates = nav_for_merge[['F_INFO_WINDCODE', 'PRICE_DATE']].assign(IS_ORIGINAL_FUND_DATE=True)

def align_fund_to_trading_days(g):
    fund_code = g['F_INFO_WINDCODE'].iloc[0]
    fund_days = trading_days[(trading_days >= g['PRICE_DATE'].min()) & (trading_days <= g['PRICE_DATE'].max())]
    return (
        g.set_index('PRICE_DATE')
         .reindex(fund_days)
         .ffill()
         .assign(F_INFO_WINDCODE=fund_code)
         .rename_axis('PRICE_DATE')
         .reset_index()
    )

nav_aligned = (
    nav_for_merge.groupby('F_INFO_WINDCODE', group_keys=False)
                 .apply(align_fund_to_trading_days)
                 .reset_index(drop=True)
)

# 标记哪些基金-日期是原始存在的，哪些是按前一交易日向前填充出来的
nav_aligned = nav_aligned.merge(
    original_fund_dates,
    on=['F_INFO_WINDCODE', 'PRICE_DATE'],
    how='left'
)
nav_aligned['IS_ORIGINAL_FUND_DATE'] = nav_aligned['IS_ORIGINAL_FUND_DATE'].fillna(False)
nav_aligned['IS_FILLED_FROM_PREV_DAY'] = ~nav_aligned['IS_ORIGINAL_FUND_DATE']

# 各类日频解释变量也统一到交易日日期列；先保证每个交易日只有一行，避免合并后行数膨胀
fm_daily = fm.rename(columns={'日期': 'PRICE_DATE'}).sort_values('PRICE_DATE').drop_duplicates('PRICE_DATE', keep='last')
fi_daily = fi.rename(columns={'TRADE_DT': 'PRICE_DATE'}).sort_values('PRICE_DATE').drop_duplicates('PRICE_DATE', keep='last')
fs_daily = fs.rename(columns={'time': 'PRICE_DATE'}).sort_values('PRICE_DATE').drop_duplicates('PRICE_DATE', keep='last')

factor_data = (
    fm_daily.merge(fi_daily, on='PRICE_DATE', how='left', validate='one_to_one')
            .merge(fs_daily, on='PRICE_DATE', how='left', validate='one_to_one')
)

# 最终用于后续回归的数据变量：基金面板日期以交易日为准，基金缺失行已用前一日内容填充
rr_data = nav_aligned.merge(factor_data, on='PRICE_DATE', how='left', validate='many_to_one')

print('nav_for_merge:', nav_for_merge.shape)
print('nav_aligned:', nav_aligned.shape)
print('factor_data:', factor_data.shape)
print('rr_data:', rr_data.shape)
print('filled fund-date rows:', int(nav_aligned['IS_FILLED_FROM_PREV_DAY'].sum()))


nav_for_merge: (719856, 22)
nav_aligned: (730666, 24)
factor_data: (244, 49)
rr_data: (730666, 72)
filled fund-date rows: 10810


In [ ]:
# 缩减不必要列
rr_data = rr_data.drop(columns=['level_0', 'index','ANN_DATE',
       'F_NAV_UNIT', 'F_NAV_DIVACCUMULATED', 'F_NAV_ADJFACTOR',
       'F_PRT_NETASSET', 'F_ASSET_MERGEDSHARESORNOT', 'NETASSET_TOTAL',
       'F_NAV_ADJUSTED', 'IS_EXDIVIDENDDATE', 'F_NAV_DISTRIBUTION',
       'S_INFO_ASHARECODE', 'CUM_NET_ASSET_VALUE'])


In [63]:

rr_data.columns

Index(['PRICE_DATE', 'level_0', 'index', 'F_INFO_WINDCODE', 'main_type', '1M',
       'return', 'SOURCE_PRICE_DATE', 'MATCHED_FM_DATE', 'DATE_SHIFT_DAYS',
       'IS_ORIGINAL_FUND_DATE', 'IS_FILLED_FROM_PREV_DAY', '上证50dailyreturn',
       '中证500dailyreturn', '中证800dailyreturn', '中证1000dailyreturn',
       '创业板指dailyreturn', '沪深300dailyreturn', '交通运输', '传媒', '公用事业', '农林牧渔',
       '化工', '医药生物', '商贸零售', '国防军工', '家用电器', '建筑材料', '建筑装饰', '房地产', '有色金属',
       '机械设备', '汽车', '煤炭', '环保', '电力设备', '电子', '石油石化', '社会服务', '纺织服饰', '综合',
       '美容护理', '计算机', '轻工制造', '通信', '采掘', '钢铁', '银行', '非银金融', '食品饮料', 'Beta',
       'BooktoPrice', 'EarningYield', 'Growth', 'Leverage', 'Liquidity',
       'Momentum', 'NonlinearSize', 'ResidualVolatility', 'Size'],
      dtype='object')

In [40]:
# 期间基金存续日期和填充时期比例
fund_fill_stats = (
    rr_data
    .groupby('F_INFO_WINDCODE')
    .agg(
        total_days=('PRICE_DATE', 'size'),
        filled_days=('IS_FILLED_FROM_PREV_DAY', 'sum'),
        fill_ratio=('IS_FILLED_FROM_PREV_DAY', 'mean')
    )
    .sort_values('fill_ratio', ascending=False)
)

fund_fill_stats.head(20)

,total_days,filled_days,fill_ratio
F_INFO_WINDCODE,,,
005549.OF,244,115,0.471311
009447.OF,236,111,0.470339
009791.OF,232,104,0.448276
166027.SZ,232,104,0.448276
161040.SZ,234,104,0.444444
506001.SH,220,97,0.440909
160325.SZ,226,98,0.433628
506006.SH,224,97,0.433036
506005.SH,223,96,0.430493


In [ ]:
# 填充超过5%的基金
fund_fill_stats[
    fund_fill_stats['fill_ratio'] > 0.05
]


,total_days,filled_days,fill_ratio
F_INFO_WINDCODE,,,
005549.OF,244,115,0.471311
009447.OF,236,111,0.470339
009791.OF,232,104,0.448276
166027.SZ,232,104,0.448276
161040.SZ,234,104,0.444444
...,...,...,...
008842.OF,195,30,0.153846
860006.OF,241,37,0.153527
010109.OF,190,29,0.152632


In [42]:
# 剔除填充比例超过 5% 的基金
funds_to_remove = fund_fill_stats.loc[fund_fill_stats['fill_ratio'] > 0.05].index
rr_data = rr_data[~rr_data['F_INFO_WINDCODE'].isin(funds_to_remove)].copy()
print('removed funds:', len(funds_to_remove))

removed funds: 285


In [43]:
# # 检查每只基金连续 ffill 的最长次数

# ffill_streak = (
#     nav_aligned
#     .sort_values(['F_INFO_WINDCODE', 'PRICE_DATE'])
#     .copy()
# )

# # True 表示这一行是 ffill 出来的
# mask = ffill_streak['IS_FILLED_FROM_PREV_DAY']

# # 给连续段编号
# ffill_streak['GROUP'] = (
#     mask.ne(mask.shift())
#         .groupby(ffill_streak['F_INFO_WINDCODE'])
#         .cumsum()
# )

# # 只统计连续 ffill 段
# ffill_streak_summary = (
#     ffill_streak[mask]
#     .groupby(['F_INFO_WINDCODE', 'GROUP'])
#     .size()
#     .reset_index(name='FFILL_STREAK')
# )

# # 每只基金最长连续 ffill 次数
# max_ffill_by_fund = (
#     ffill_streak_summary
#     .groupby('F_INFO_WINDCODE')['FFILL_STREAK']
#     .max()
#     .reset_index(name='MAX_FFILL_STREAK')
#     .sort_values('MAX_FFILL_STREAK', ascending=False)
# )

# print('所有基金中最长连续 ffill 次数:')
# print(max_ffill_by_fund['MAX_FFILL_STREAK'].max())

# print('\nTop 30:')
# display(max_ffill_by_fund.head(30))

In [44]:
# # =========================
# # 每只基金真实开始日期
# # （基于原始净值数据）
# # =========================

# fund_start_dates = (
#     nav_for_merge
#     .groupby('F_INFO_WINDCODE')['PRICE_DATE']
#     .min()
#     .rename('TRUE_START_DATE')
#     .reset_index()
# )

# # =========================
# # 合并到 rr_data
# # =========================

# rr_check = rr_data.merge(
#     fund_start_dates,
#     on='F_INFO_WINDCODE',
#     how='left'
# )

# # =========================
# # 基金成立前的数据
# # =========================

# before_start_rows = rr_check[
#     rr_check['PRICE_DATE']
#     < rr_check['TRUE_START_DATE']
# ].copy()

# # =========================
# # 成立前存在 return 的行
# # =========================

# abnormal_before_start = before_start_rows[
#     before_start_rows['return'].notna()
# ]

# # =========================
# # 输出
# # =========================

# print('成立前存在 return 的行数:', len(abnormal_before_start))

# print(
#     '成立前存在 return 的基金数量:',
#     abnormal_before_start['F_INFO_WINDCODE'].nunique()
# )

# display(
#     abnormal_before_start
#     .sort_values(['F_INFO_WINDCODE', 'PRICE_DATE'])
#     .head(30)
# )

In [ ]:
# 开始回测暴露值
import pandas as pd
import numpy as np
import cvxpy as cp

# 假设你的对齐数据叫 df
df = rr_data.copy()  # 你提供的对齐数据

# 风格因子列表
style_factors = [
    'Beta', 'BooktoPrice', 'EarningYield', 'Growth', 'Leverage',
    'Liquidity', 'Momentum', 'NonlinearSize', 'ResidualVolatility', 'Size'
]

#行业因子列表
industry_factors = [
    '交通运输', '传媒', '公用事业', '农林牧渔',
    '化工', '医药生物', '商贸零售', '国防军工',
    '家用电器', '建筑材料', '建筑装饰', '房地产',
    '有色金属', '机械设备', '汽车', '煤炭',
    '环保', '电力设备', '电子', '石油石化',
    '社会服务', '纺织服饰', '综合', '美容护理',
    '计算机', '轻工制造', '通信', '采掘',
    '钢铁', '银行', '非银金融', '食品饮料'
]

# 存放回归结果
results = []

# 所有基金列表
funds = df['F_INFO_WINDCODE'].unique()

# 正则化系数
lambda_style = 6e-5

# 遍历每只基金
for fund in funds:
    fund_data = df[df['F_INFO_WINDCODE'] == fund].copy()

    if len(fund_data) < 20:
        continue

    if fund_data.empty:
        continue
    
    # y = 基金收益率
    y = fund_data['return'].values
    
    # 市场因子矩阵
    market_return = fund_data['沪深300dailyreturn'].values

    # 行业收益率矩阵
    industry_matrix = (
        fund_data[industry_factors]
        .fillna(0)
        .values
    )

    n_industry = len(industry_factors)
    
    # 风格矩阵
    X_style = fund_data[style_factors].values
    n_style = X_style.shape[1]
    
    # 持仓暴露变量（股票仓位）
    beta_fund = cp.Variable()
    
    # 行业暴露
    beta_industry = cp.Variable(n_industry)
    
    # 风格暴露
    beta_style = cp.Variable(n_style)
    
    # alpha
    alpha = cp.Variable()
    
    # 预测收益 = 持仓暴露*市场 + 行业矩阵*行业暴露 + 风格矩阵*风格暴露 + alpha
    pred = beta_fund* market_return + industry_matrix @ beta_industry + X_style @ beta_style + alpha
    
    # L2 正则只对风格暴露
    penalty = lambda_style * cp.sum_squares(beta_style)

    objective = cp.Minimize(
        cp.sum_squares(y - pred)
        + penalty
    )

    
    # 目标函数
    objective = cp.Minimize(cp.sum_squares(y - pred) + penalty)
    
    # 约束条件
    constraints = [

        beta_fund >= 0,
        beta_fund <= 1,

        beta_industry >= 0,
        beta_industry <= 1,

        cp.sum(beta_industry) == beta_fund
    ]
    
    # 构建问题并求解
    problem = cp.Problem(objective, constraints)

    problem.solve(
        solver=cp.ECOS,
        verbose=False
    )

    if beta_fund.value is None:
        continue
    
    # 保存结果
    results.append({

        "F_INFO_WINDCODE": fund,

        "stock_exposure": float(beta_fund.value),

        "alpha": float(alpha.value),

        "industry_exposure":
            dict(zip(
                industry_factors,
                beta_industry.value
            )),

        "style_exposure":
            dict(zip(
                style_factors,
                beta_style.value
            ))
    })

# 转为 DataFrame 查看
results_df = pd.DataFrame(results)

# 展示前几条
print(results_df.head())

  F_INFO_WINDCODE  stock_exposure     alpha  \
0       000001.OF        0.466520 -0.000334   
1       000006.OF        0.473632  0.000729   
2       000011.OF        0.603221  0.000241   
3       000017.OF        0.491445 -0.000104   
4       000020.OF        0.404499  0.000101   

                                   industry_exposure  \
0  {'交通运输': 5.980539846340478e-09, '传媒': 4.496879...   
1  {'交通运输': 3.497878072861086e-09, '传媒': 3.009190...   
2  {'交通运输': 1.6530027147718127e-08, '传媒': 3.59933...   
3  {'交通运输': 3.147679849473535e-09, '传媒': 3.322821...   
4  {'交通运输': 2.1831825909947455e-08, '传媒': 3.69865...   

                                      style_exposure  
0  {'Beta': -0.1703156304342712, 'BooktoPrice': -...  
1  {'Beta': -0.4919149021833258, 'BooktoPrice': 0...  
2  {'Beta': -1.3548421031780185, 'BooktoPrice': -...  
3  {'Beta': 0.24378444414913752, 'BooktoPrice': -...  
4  {'Beta': 0.5636738899949786, 'BooktoPrice': -0...  


In [46]:
# end_date 单日快照：用最新可用持仓计算 rr_data 基金的 Barra 正交后风格暴露
import gc
import numpy as np
import pandas as pd

style_factor_files = {
    'Beta': 'Beta正交后.txt',
    'BooktoPrice': 'BooktoPrice正交后.txt',
    'EarningYield': 'EarningYield正交后.txt',
    'Growth': 'Growth正交后.txt',
    'Leverage': 'Leverage正交后.txt',
    'Liquidity': 'Liquidity正交后.txt',
    'Momentum': 'Momentum正交后.txt',
    'NonlinearSize': 'NonlinearSize正交后.txt',
    'ResidualVolatility': 'ResidualVolatility正交后.txt',
    'Size': 'Size正交后.txt',
}

rr_fund_col = 'F_INFO_WINDCODE'
sh_fund_col = 'S_INFO_WINDCODE'
report_col = 'F_PRT_ENDDATE'
available_col = 'available_date'
stock_col = 'S_INFO_STOCKWINDCODE'
weight_col = 'F_PRT_STKVALUETONAV'

required_cols = [sh_fund_col, report_col, available_col, stock_col, weight_col]
missing_cols = [col for col in required_cols if col not in SH.columns]
if missing_cols:
    raise KeyError(f'SH 缺少必要列: {missing_cols}')

snapshot_date = pd.to_datetime(end_date)
if rr_fund_col not in rr_data.columns:
    raise KeyError(f'rr_data 缺少基金代码列 {rr_fund_col}')
target_funds = pd.Index(rr_data[rr_fund_col].dropna().unique())
if len(target_funds) == 0:
    raise ValueError(f'rr_data 当前行数为 {len(rr_data)}，但 {rr_fund_col} 没有可用基金代码。请检查前面的筛选步骤。')

# 先用全量 SH 判断 end_date 当天最新可用的半年/全年持仓期；不要先按 rr_data 基金过滤，
# 否则 rr_data 为空或基金交集异常时会误报“没有可用持仓”。
holding_all = SH.loc[:, required_cols].copy()
holding_all[report_col] = pd.to_datetime(holding_all[report_col], errors='coerce')
holding_all[available_col] = pd.to_datetime(holding_all[available_col], errors='coerce')
holding_all[weight_col] = pd.to_numeric(holding_all[weight_col], errors='coerce')

semi_annual_or_annual = (
    ((holding_all[report_col].dt.month == 6) & (holding_all[report_col].dt.day == 30))
    | ((holding_all[report_col].dt.month == 12) & (holding_all[report_col].dt.day == 31))
)

holding_available_all = holding_all.loc[
    semi_annual_or_annual
    & holding_all[available_col].notna()
    & (holding_all[available_col] <= snapshot_date)
    & holding_all[report_col].notna()
].copy()

if holding_available_all.empty:
    raise ValueError(f'end_date={snapshot_date.date()} 前没有可用的半年/全年持仓。')

latest_holding = (
    holding_available_all[[report_col, available_col]]
    .drop_duplicates()
    .sort_values([available_col, report_col])
    .iloc[-1]
)
latest_report_date = latest_holding[report_col]
latest_available_date = latest_holding[available_col]

holding_snapshot_all = holding_available_all.loc[
    (holding_available_all[report_col] == latest_report_date)
    & (holding_available_all[available_col] == latest_available_date)
    & holding_available_all[stock_col].notna()
    & holding_available_all[weight_col].gt(0)
].copy()

holding_snapshot = holding_snapshot_all.loc[
    holding_snapshot_all[sh_fund_col].isin(target_funds)
].copy()

if holding_snapshot.empty:
    sh_sample = holding_snapshot_all[sh_fund_col].dropna().astype(str).head(5).tolist()
    rr_sample = pd.Series(target_funds).dropna().astype(str).head(5).tolist()
    raise ValueError(
        f'最新可用持仓期 {latest_report_date.date()} / {latest_available_date.date()} 存在，'
        f'但与 rr_data 基金没有匹配。SH基金样例={sh_sample}，rr_data基金样例={rr_sample}。'
    )

if holding_snapshot.empty:
    raise ValueError(
        f'最新可用持仓 {latest_report_date.date()} / {latest_available_date.date()} 没有正权重股票记录。'
    )

# 同一基金同一股票可能有多条记录，先把权重加总；权重单位由百分比转成小数。
holding_snapshot['holding_weight'] = holding_snapshot[weight_col] / 100.0
holding_snapshot = (
    holding_snapshot
    .groupby([sh_fund_col, stock_col], as_index=False)['holding_weight']
    .sum()
)

holding_stock_weight_sum = (
    holding_snapshot.groupby(sh_fund_col, as_index=False)['holding_weight']
    .sum()
    .rename(columns={'holding_weight': 'holding_stock_weight_sum'})
)

# Barra 暴露日期：用 end_date 当天或之前最近一个可用日期。
beta_all = pd.read_feather(fold / 'Barra_CNE5' / style_factor_files['Beta'])
beta_all['time'] = pd.to_datetime(beta_all['time'])
barra_dates = beta_all.loc[beta_all['time'] <= snapshot_date, 'time'].dropna()
if barra_dates.empty:
    raise ValueError(f'end_date={snapshot_date.date()} 前没有可用的 Barra 暴露日期。')
barra_exposure_date = barra_dates.max()
del beta_all
gc.collect()

needed_stocks = pd.Index(holding_snapshot[stock_col].dropna().unique())
style_exposure = pd.DataFrame({sh_fund_col: sorted(holding_snapshot[sh_fund_col].dropna().unique())})
matched_stock_weight_sum = None

for factor_name, file_name in style_factor_files.items():
    factor_wide = pd.read_feather(fold / 'Barra_CNE5' / file_name)
    factor_wide['time'] = pd.to_datetime(factor_wide['time'])

    available_stocks = [stock for stock in needed_stocks if stock in factor_wide.columns]
    factor_row = factor_wide.loc[factor_wide['time'] == barra_exposure_date, available_stocks]
    if factor_row.empty:
        raise ValueError(f'{file_name} 找不到 Barra 日期 {barra_exposure_date.date()}。')

    factor_long = factor_row.iloc[0].rename(factor_name).reset_index()
    factor_long.columns = [stock_col, factor_name]
    factor_long[factor_name] = pd.to_numeric(factor_long[factor_name], errors='coerce')
    factor_long = factor_long.dropna(subset=[factor_name])

    weighted = holding_snapshot.merge(factor_long, on=stock_col, how='inner')
    weighted[factor_name] = weighted['holding_weight'] * weighted[factor_name]

    factor_exposure = weighted.groupby(sh_fund_col, as_index=False)[factor_name].sum()
    style_exposure = style_exposure.merge(factor_exposure, on=sh_fund_col, how='left')

    # 10 个正交后风格文件的股票覆盖列通常一致，覆盖权重统计一次即可。
    if matched_stock_weight_sum is None:
        matched_stock_weight_sum = (
            weighted.groupby(sh_fund_col, as_index=False)['holding_weight']
            .sum()
            .rename(columns={'holding_weight': 'matched_stock_weight_sum'})
        )

    del factor_wide, factor_row, factor_long, weighted, factor_exposure
    gc.collect()
    print(f'{factor_name} done')

holding_style_exposure = (
    style_exposure
    .rename(columns={sh_fund_col: rr_fund_col})
    .merge(
        holding_stock_weight_sum.rename(columns={sh_fund_col: rr_fund_col}),
        on=rr_fund_col,
        how='left',
    )
    .merge(
        matched_stock_weight_sum.rename(columns={sh_fund_col: rr_fund_col}),
        on=rr_fund_col,
        how='left',
    )
    .assign(
        PRICE_DATE=snapshot_date,
        holding_report_date=latest_report_date,
        holding_available_date=latest_available_date,
        barra_exposure_date=barra_exposure_date,
    )
)

holding_style_exposure = holding_style_exposure[
    [
        rr_fund_col,
        'PRICE_DATE',
        'holding_report_date',
        'holding_available_date',
        'barra_exposure_date',
        'holding_stock_weight_sum',
        'matched_stock_weight_sum',
    ]
    + list(style_factor_files.keys())
].sort_values(rr_fund_col).reset_index(drop=True)

# 兼容前面命名：这个结果是一只基金一行的 end_date 快照，不是每日滚动面板。
holding_style_exposure_panel = holding_style_exposure

rr_data_with_holding_style = rr_data.merge(
    holding_style_exposure.drop(columns=['PRICE_DATE']),
    on=rr_fund_col,
    how='left',
    validate='many_to_one',
)

coverage = (
    holding_style_exposure['matched_stock_weight_sum']
    / holding_style_exposure['holding_stock_weight_sum']
)

print('snapshot_date:', snapshot_date.date())
print('latest holding report date:', latest_report_date.date())
print('latest holding available date:', latest_available_date.date())
print('barra exposure date:', barra_exposure_date.date())
print('matched / holding stock weight coverage:')
print(coverage.describe())
display(holding_style_exposure.head())


Beta done
BooktoPrice done
EarningYield done
Growth done
Leverage done
Liquidity done
Momentum done
NonlinearSize done
ResidualVolatility done
Size done
snapshot_date: 2021-06-29
latest holding report date: 2020-12-31
latest holding available date: 2021-03-31
barra exposure date: 2021-06-29
matched / holding stock weight coverage:
count    2736.000000
mean        0.943648
std         0.127310
min         0.000547
25%         0.954913
50%         0.987192
75%         1.000000
max         1.000000
dtype: float64


,F_INFO_WINDCODE,PRICE_DATE,holding_report_date,holding_available_date,barra_exposure_date,holding_stock_weight_sum,matched_stock_weight_sum,Beta,BooktoPrice,EarningYield,Growth,Leverage,Liquidity,Momentum,NonlinearSize,ResidualVolatility,Size
0,000001.OF,2021-06-29,2020-12-31,2021-03-31,2021-06-29,0.7520,0.7330,-0.053251,-0.170641,0.296354,0.086783,-0.017659,0.161759,-0.185330,0.170413,-0.114264,0.147049
1,000006.OF,2021-06-29,2020-12-31,2021-03-31,2021-06-29,0.9136,0.8929,0.457725,-0.270160,-0.120176,0.155225,-0.088839,0.149667,0.151418,0.424853,0.054752,-0.263712
2,000011.OF,2021-06-29,2020-12-31,2021-03-31,2021-06-29,0.9149,0.8852,0.210509,-0.421723,-0.302746,0.259963,0.383667,-0.380383,0.223271,0.076787,0.261663,-0.054705
3,000017.OF,2021-06-29,2020-12-31,2021-03-31,2021-06-29,0.9160,0.9014,0.824535,-0.669289,-0.464174,0.033252,-0.201521,-0.212935,0.190572,-0.565232,-0.002391,0.288311
4,000020.OF,2021-06-29,2020-12-31,2021-03-31,2021-06-29,0.8823,0.8820,0.629999,-0.692565,-0.435938,0.162270,-0.256184,-0.049344,-0.067615,-0.079163,0.070031,0.108839


In [47]:
# 回归风格暴露与持仓风格暴露等权混合，重算模型收益和选股能力
import numpy as np
import pandas as pd

# 检查数据是否存在，并且是否为空
required_objects = ['results_df', 'holding_style_exposure', 'rr_data']
for obj_name in required_objects:
    if obj_name not in globals():
        raise NameError(f'请先运行生成 {obj_name} 的前置 cell。')

if results_df.empty:
    raise ValueError('results_df 为空，请先成功运行回归暴露 cell。')
if holding_style_exposure.empty:
    raise ValueError('holding_style_exposure 为空，请先成功运行持仓 Barra 暴露 cell。')

# 定义因子名
style_factors = [
    'Beta', 'BooktoPrice', 'EarningYield', 'Growth', 'Leverage',
    'Liquidity', 'Momentum', 'NonlinearSize', 'ResidualVolatility', 'Size'
]

industry_factors = [
    '交通运输', '传媒', '公用事业', '农林牧渔',
    '化工', '医药生物', '商贸零售', '国防军工',
    '家用电器', '建筑材料', '建筑装饰', '房地产',
    '有色金属', '机械设备', '汽车', '煤炭',
    '环保', '电力设备', '电子', '石油石化',
    '社会服务', '纺织服饰', '综合', '美容护理',
    '计算机', '轻工制造', '通信', '采掘',
    '钢铁', '银行', '非银金融', '食品饮料'
]
# 定义其他列名
fund_col = 'F_INFO_WINDCODE'
market_col = '沪深300dailyreturn'
return_col = 'return'
# 检查rr_data中是否存在相关列
required_rr_cols = [fund_col, 'PRICE_DATE', return_col, market_col] + style_factors + industry_factors
missing_rr_cols = [col for col in required_rr_cols if col not in rr_data.columns]
if missing_rr_cols:
    raise KeyError(f'rr_data 缺少计算模型收益所需列: {missing_rr_cols}')
# 检查result_df中是否包含相关结果
missing_result_cols = [col for col in [fund_col, 'stock_exposure', 'style_exposure', 'industry_exposure'] if col not in results_df.columns]
if missing_result_cols:
    raise KeyError(f'results_df 缺少必要列: {missing_result_cols}')
# 检查十个风格因子是否都存在
missing_holding_cols = [col for col in [fund_col] + style_factors if col not in holding_style_exposure.columns]
if missing_holding_cols:
    raise KeyError(f'holding_style_exposure 缺少必要列: {missing_holding_cols}')

# 展开回归得到的风格暴露和行业暴露。
reg_style_exposure = pd.json_normalize(results_df['style_exposure']).reindex(columns=style_factors)
reg_style_exposure.columns = [f'{col}_reg_style_exposure' for col in style_factors]
reg_style_exposure.insert(0, fund_col, results_df[fund_col].values)

reg_industry_exposure = pd.json_normalize(results_df['industry_exposure']).reindex(columns=industry_factors).fillna(0.0)
reg_industry_exposure.columns = [f'{col}_industry_exposure' for col in industry_factors]
reg_industry_exposure.insert(0, fund_col, results_df[fund_col].values)

reg_base_exposure = results_df[[fund_col, 'stock_exposure']].copy()

holding_style = holding_style_exposure[[fund_col] + style_factors].copy()
holding_style = holding_style.rename(columns={col: f'{col}_holding_style_exposure' for col in style_factors})
# 合并所有暴露信息
mix_style_exposure = (
    reg_base_exposure
    .merge(reg_style_exposure, on=fund_col, how='inner', validate='one_to_one')
    .merge(reg_industry_exposure, on=fund_col, how='inner', validate='one_to_one')
    .merge(holding_style, on=fund_col, how='inner', validate='one_to_one')
)
# 计算混合风格暴露
for factor in style_factors:
    mix_style_exposure[f'{factor}_mix_style_exposure'] = (
        mix_style_exposure[f'{factor}_reg_style_exposure']
        + mix_style_exposure[f'{factor}_holding_style_exposure']
    ) / 2.0

# 用 mix 风格暴露带入收益模型：市场 + 行业 + mix风格 + alpha。
model_cols = (
    [fund_col, 'stock_exposure']
    + [f'{factor}_mix_style_exposure' for factor in style_factors]
    + [f'{factor}_reg_style_exposure' for factor in style_factors]
    + [f'{factor}_holding_style_exposure' for factor in style_factors]
    + [f'{industry}_industry_exposure' for industry in industry_factors]
)
# 暴露信息并入到每日数据中
rr_data_with_mix_exposure = rr_data.merge(
    mix_style_exposure[model_cols],
    on=fund_col,
    how='inner',
    validate='many_to_one',
)

if rr_data_with_mix_exposure.empty:
    raise ValueError('rr_data 与 mix_style_exposure 合并后为空，请检查基金代码交集。')
# 计算市场收益贡献
market_model_return = (
    rr_data_with_mix_exposure['stock_exposure']
    * rr_data_with_mix_exposure[market_col].fillna(0.0)
)
# 计算行业收益贡献
industry_model_return = np.zeros(len(rr_data_with_mix_exposure), dtype=float)
for industry in industry_factors:
    industry_model_return += (
        rr_data_with_mix_exposure[industry].fillna(0.0).to_numpy(dtype=float)
        * rr_data_with_mix_exposure[f'{industry}_industry_exposure'].fillna(0.0).to_numpy(dtype=float)
    )
# 计算风格收益贡献
style_model_return = np.zeros(len(rr_data_with_mix_exposure), dtype=float)
for factor in style_factors:
    style_model_return += (
        rr_data_with_mix_exposure[factor].fillna(0.0).to_numpy(dtype=float)
        * rr_data_with_mix_exposure[f'{factor}_mix_style_exposure'].fillna(0.0).to_numpy(dtype=float)
    )

rr_data_with_mix_exposure['market_model_return'] = market_model_return
rr_data_with_mix_exposure['industry_model_return'] = industry_model_return
rr_data_with_mix_exposure['style_model_return_mix'] = style_model_return
rr_data_with_mix_exposure['model_return_mix'] = (
    rr_data_with_mix_exposure['market_model_return']
    + rr_data_with_mix_exposure['industry_model_return']
    + rr_data_with_mix_exposure['style_model_return_mix']
    # + rr_data_with_mix_exposure['alpha'] 不需要加alpha
)
rr_data_with_mix_exposure['stock_selection_return_mix'] = (
    rr_data_with_mix_exposure[return_col]
    - rr_data_with_mix_exposure['model_return_mix']
)

stock_selection_ability = (
    rr_data_with_mix_exposure
    .groupby(fund_col, as_index=False)
    .agg(
        stock_selection_ability=('stock_selection_return_mix', 'mean'),
        avg_nav_return=(return_col, 'mean'),
        avg_model_return_mix=('model_return_mix', 'mean'),
        obs_count=('stock_selection_return_mix', 'count'),
    )
    .sort_values('stock_selection_ability', ascending=False)
    .reset_index(drop=True)
)

final_stock_selection_ability = stock_selection_ability['stock_selection_ability'].mean()

print('regression funds:', results_df[fund_col].nunique())
print('holding exposure funds:', holding_style_exposure[fund_col].nunique())
print('mix exposure funds:', mix_style_exposure[fund_col].nunique())
display(mix_style_exposure[[fund_col] + [f'{factor}_mix_style_exposure' for factor in style_factors]].head())
display(stock_selection_ability.head(15))


regression funds: 2771
holding exposure funds: 2745
mix exposure funds: 2745


,F_INFO_WINDCODE,Beta_mix_style_exposure,BooktoPrice_mix_style_exposure,EarningYield_mix_style_exposure,Growth_mix_style_exposure,Leverage_mix_style_exposure,Liquidity_mix_style_exposure,Momentum_mix_style_exposure,NonlinearSize_mix_style_exposure,ResidualVolatility_mix_style_exposure,Size_mix_style_exposure
0,000001.OF,-0.111784,-0.108384,-0.045059,0.352538,-0.515969,-0.106372,0.038160,-0.171711,0.066252,0.122472
1,000006.OF,-0.017095,-0.031663,-0.373771,0.597639,-0.563496,-0.206841,0.501258,0.954817,0.517411,-0.522210
2,000011.OF,-0.572167,-0.801531,0.366071,0.982485,0.482887,-0.343139,0.153379,0.529723,-0.412841,-0.415446
3,000017.OF,0.534160,-0.366711,-0.817001,0.273827,-0.510778,-0.772439,0.259510,-0.467181,0.402732,0.145675
4,000020.OF,0.596836,-0.426465,-0.457456,0.580777,-0.678675,-0.142892,0.148841,-0.118038,-0.192854,0.038283


,F_INFO_WINDCODE,stock_selection_ability,avg_nav_return,avg_model_return_mix,obs_count
0,003834.OF,0.002067,0.003858,0.001791,244
1,005669.OF,0.001996,0.003854,0.001858,244
2,009147.OF,0.001935,0.003804,0.001868,244
3,007192.OF,0.001828,0.003574,0.001746,244
4,010685.OF,0.001823,0.003422,0.001598,144
5,005928.OF,0.001783,0.003504,0.001722,244
6,006299.OF,0.001737,0.003532,0.001795,244
7,005927.OF,0.001736,0.003513,0.001777,244
8,001487.OF,0.001728,0.002502,0.000774,244
9,400015.OF,0.001722,0.003821,0.002100,244


In [48]:
# 选股能力 Top15：只做基金经理去重，不限制基金公司
import numpy as np
import pandas as pd

if 'stock_selection_ability' not in globals():
    raise NameError('请先运行生成 stock_selection_ability 的前置 cell。')
if stock_selection_ability.empty:
    raise ValueError('stock_selection_ability 为空。')

manager_path = fold / '基金数据' / 'CHINAMUTUALFUNDMANAGER_202605221351(1).csv'
manager_raw = pd.read_csv(manager_path)

fund_col = 'F_INFO_WINDCODE'
manager_name_col = 'F_INFO_FUNDMANAGER'
manager_id_col = 'F_INFO_FUNDMANAGER_ID'
manager_start_col = 'F_INFO_MANAGER_STARTDATE'
manager_leave_col = 'F_INFO_MANAGER_LEAVEDATE'

required_manager_cols = [fund_col, manager_name_col, manager_id_col, manager_start_col, manager_leave_col]
missing_manager_cols = [col for col in required_manager_cols if col not in manager_raw.columns]
if missing_manager_cols:
    raise KeyError(f'基金经理表缺少必要列: {missing_manager_cols}')

def _parse_yyyymmdd(series):
    raw = series.astype('string').str.replace(r'\.0$', '', regex=True)
    raw = raw.mask(raw.isin(['0', 'nan', '<NA>', 'None', '']))
    return pd.to_datetime(raw, format='%Y%m%d', errors='coerce')

snapshot_date = pd.to_datetime(end_date)
manager = manager_raw[required_manager_cols].copy()
manager[manager_start_col] = _parse_yyyymmdd(manager[manager_start_col])
manager[manager_leave_col] = _parse_yyyymmdd(manager[manager_leave_col])
manager[manager_id_col] = manager[manager_id_col].astype('string').str.replace(r'\.0$', '', regex=True).str.zfill(5)
manager[manager_name_col] = manager[manager_name_col].astype('string')

# 优先取 end_date 当天仍在任的基金经理；如果某基金没有在任记录，则用 end_date 前最近一条任职记录兜底。
active_manager = manager.loc[
    manager[manager_start_col].notna()
    & (manager[manager_start_col] <= snapshot_date)
    & (manager[manager_leave_col].isna() | (manager[manager_leave_col] >= snapshot_date))
].copy()
active_manager['manager_source'] = 'active_on_end_date'

funds_with_active = set(active_manager[fund_col].dropna())
fallback_manager = manager.loc[
    manager[fund_col].isin(stock_selection_ability[fund_col])
    & ~manager[fund_col].isin(funds_with_active)
    & manager[manager_start_col].notna()
    & (manager[manager_start_col] <= snapshot_date)
].copy()

if not fallback_manager.empty:
    fallback_manager = (
        fallback_manager
        .sort_values([fund_col, manager_start_col, manager_leave_col], ascending=[True, False, False])
        .drop_duplicates(fund_col, keep='first')
    )
    fallback_manager['manager_source'] = 'latest_before_end_date'

manager_effective = pd.concat([active_manager, fallback_manager], ignore_index=True)
manager_effective = manager_effective.loc[manager_effective[fund_col].isin(stock_selection_ability[fund_col])].copy()

# 一只基金可能有多位共同基金经理：保留完整经理组；只要经理组里任一经理已出现，后面的基金就跳过。
manager_info = (
    manager_effective
    .dropna(subset=[fund_col, manager_id_col])
    .groupby(fund_col)
    .agg(
        manager_ids=(manager_id_col, lambda x: tuple(sorted(set(x.dropna().astype(str))))),
        manager_names=(manager_name_col, lambda x: '、'.join(sorted(set(x.dropna().astype(str))))),
        manager_source=('manager_source', lambda x: 'active_on_end_date' if (x == 'active_on_end_date').any() else 'latest_before_end_date'),
    )
    .reset_index()
)

ranked = stock_selection_ability.sort_values('stock_selection_ability', ascending=False).reset_index(drop=True).copy()
ranked = ranked.merge(manager_info, on=fund_col, how='left', validate='one_to_one')
ranked['original_rank'] = np.arange(1, len(ranked) + 1)

selected_rows = []
removed_rows = []
used_manager_ids = set()

for _, row in ranked.iterrows():
    manager_ids = row.get('manager_ids')
    if not isinstance(manager_ids, tuple) or len(manager_ids) == 0:
        removed = row.copy()
        removed['remove_reason'] = 'missing_manager'
        removed_rows.append(removed)
        continue

    repeated = sorted(set(manager_ids).intersection(used_manager_ids))
    if repeated:
        removed = row.copy()
        removed['remove_reason'] = 'duplicate_manager'
        removed['duplicate_manager_ids'] = '|'.join(repeated)
        removed_rows.append(removed)
        continue

    selected_rows.append(row)
    used_manager_ids.update(manager_ids)

    if len(selected_rows) >= 15:
        break

stock_selection_ability_top15_unique_manager = pd.DataFrame(selected_rows).reset_index(drop=True)
removed_by_manager_rule = pd.DataFrame(removed_rows).reset_index(drop=True)

if stock_selection_ability_top15_unique_manager.empty:
    raise ValueError('基金经理去重后没有选出基金，请检查基金经理表和基金代码匹配。')

stock_selection_ability_top15_unique_manager['selected_rank'] = np.arange(1, len(stock_selection_ability_top15_unique_manager) + 1)

print('snapshot_date:', snapshot_date.date())
print('selected funds:', len(stock_selection_ability_top15_unique_manager))
print('unique manager ids:', len(used_manager_ids))
print('removed before filling top15:', len(removed_by_manager_rule))
if len(stock_selection_ability_top15_unique_manager) < 15:
    print('WARNING: 去重后不足 15 只基金。')

display(
    stock_selection_ability_top15_unique_manager[
        ['selected_rank', 'original_rank', fund_col, 'stock_selection_ability', 'manager_names', 'manager_ids', 'manager_source']
    ]
)

display(
    removed_by_manager_rule[
        [col for col in ['original_rank', fund_col, 'stock_selection_ability', 'manager_names', 'manager_ids', 'remove_reason', 'duplicate_manager_ids'] if col in removed_by_manager_rule.columns]
    ].head(30)
)


snapshot_date: 2021-06-29
selected funds: 15
unique manager ids: 18
removed before filling top15: 4


,selected_rank,original_rank,F_INFO_WINDCODE,stock_selection_ability,manager_names,manager_ids,manager_source
0,1,1,003834.OF,0.002067,郑泽鸿,"(7AF40D,)",active_on_end_date
1,2,2,005669.OF,0.001996,崔宸龙,"(14383E9,)",active_on_end_date
2,3,3,009147.OF,0.001935,张湘龙、田元泉、陶灿,"(01116, 145656D, JR14E336E)",active_on_end_date
3,4,4,007192.OF,0.001828,高楠,"(140B23F,)",active_on_end_date
4,5,5,010685.OF,0.001823,赵蓓,"(7A0E47,)",active_on_end_date
5,6,6,005928.OF,0.001783,曹春林,"(161CB4,)",active_on_end_date
6,7,9,001487.OF,0.001728,肖肖、陈金伟,"(170622, JR14DB684)",active_on_end_date
7,8,10,400015.OF,0.001722,李瑞,"(7C633D,)",active_on_end_date
8,9,11,290011.OF,0.001700,董季周,"(7BCE05,)",active_on_end_date
9,10,12,001951.OF,0.001699,韩广哲,"(16F8D9,)",active_on_end_date


,original_rank,F_INFO_WINDCODE,stock_selection_ability,manager_names,manager_ids,remove_reason,duplicate_manager_ids
0,7,006299.OF,0.001737,叶佳、高楠,"(140B23F, JR14AEF5A)",duplicate_manager,140B23F
1,8,005927.OF,0.001736,曹春林,"(161CB4,)",duplicate_manager,161CB4
2,17,002084.OF,0.001601,刘彬,"(16DE02,)",duplicate_manager,16DE02
3,18,001644.OF,0.001585,陆彬,"(7D6EBB,)",duplicate_manager,7D6EBB


In [61]:
stock_selection_ability[stock_selection_ability['F_INFO_WINDCODE']=='519674.OF']

,F_INFO_WINDCODE,stock_selection_ability,avg_nav_return,avg_model_return_mix,obs_count
364,519674.OF,0.000593,0.001267,0.000674,244
